<a href="https://colab.research.google.com/github/bahmedx/730/blob/main/stock_stochastic_simulation2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Section 1: Introduction**

# Modeling Stock Price Movements Using Stochastic Processes

This project compares three stochastic models:

1. Geometric Brownian Motion (GBM)
2. Ornstein-Uhlenbeck Process (OU)
3. Merton Jump-Diffusion

Stocks analyzed:

- Apple (AAPL)
- Microsoft (MSFT)
- Nvidia (NVDA)

Monte Carlo simulations are used to forecast future stock prices and assess investment risk.

# **Section 2: Install Packages**

In [ ]:
!pip install yfinance --quiet

# **Section 3: Import Libraries**

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import norm
from scipy.stats import jarque_bera

# **Section 4: Download Multiple Stocks**

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA"]

data = yf.download(
    tickers,
    start="2024-01-01",
    auto_adjust=True
)

prices = data["Close"]
prices.tail()

# **Section 5: Data Cleaning**

In [ ]:
prices = prices.dropna()

returns = np.log(prices / prices.shift(1))
returns = returns.dropna()

returns.head()

# **Section 6: Exploratory Analysis**

**Price History**

In [ ]:
prices.plot(
    figsize=(12,6),
    title="Historical Stock Prices"
)

plt.show()

**Correlation Matrix**

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    returns.corr(),
    annot=True,
    cmap="coolwarm"
)

plt.title("Correlation Matrix")
plt.show()

# **Section 7: Risk Statistics Table**

In [ ]:
summary = pd.DataFrame()

summary["Mean Return["AAPL", "MSFT", "NVDA"01()
ary["Volatility"] = returns.std()*np.sqrt(252)

summary["Sharpe Ratio"] = (
    summary["Mean Return"] /
    summary["Volatility"]
)

summary

# **Section 8: GBM Model**
Use exact SDE:

dSt=μStdt+σStdWtdS_t = \mu S_t dt + \sigma S_t dW_tdSt​=μSt​dt+σSt​dWt​

In [ ]:
def gbm_simulation(
    S0,
    mu,
    sigma,
    days,
    simulations
):

    dt = 1/252

    paths = np.zeros((days, simulations))
    paths[0] = S0

    for t in range(1, days):

        z = np.random.standard_normal(simulations)

        paths[t] = (
            paths[t-1]
            * np.exp(
                (mu-0.5*sigma**2)*dt
                + sigma*np.sqrt(dt)*z
            )
        )

    return paths

# **Section 9: Ornstein-Uhlenbeck**

In [ ]:
def ou_simulation(
    x0,
    theta,
    mu,
    sigma,
    days,
    simulations
):

    dt = 1/252

    paths = np.zeros((days, simulations))
    paths[0] = x0

    for t in range(1, days):

        z = np.random.normal(
            0,
            1,
            simulations
        )

        paths[t] = (
            paths[t-1]
            + theta*(mu-paths[t-1])*dt
            + sigma*np.sqrt(dt)*z
        )

    return paths

# **Section 10: Merton Jump-Diffusion**
dS=μSdt+σSdW+Jdq

In [ ]:
def jump_diffusion(
    S0,
    mu,
    sigma,
    lam,
    jump_mu,
    jump_sigma,
    days,
    simulations
):

    dt = 1/252

    paths = np.zeros((days, simulations))
    paths[0] = S0

    for t in range(1, days):

        z = np.random.normal(
            0,
            1,
            simulations
        )

        poisson = np.random.poisson(
            lam*dt,
            simulations
        )

        jumps = (
            poisson
            * np.random.normal(
                jump_mu,
                jump_sigma,
                simulations
            )
        )

        paths[t] = (
            paths[t-1]
            * np.exp(
                (mu-0.5*sigma**2)*dt
                + sigma*np.sqrt(dt)*z
                + jumps
            )
        )

    return paths

# **Section 11: 5000 Monte Carlo Simulations**

In [ ]:
SIMS = 5000
DAYS = 180

# **Section 12: Confidence Intervals**

In [ ]:
gbm_lower = np.percentile(
    gbm_final,
    2.5
)

gbm_upper = np.percentile(
    gbm_final,
    97.5
)

print("95% CI")

print(
    gbm_lower,
    gbm_upper
)

# **Section 13: Value at Risk (VaR)**

In [ ]:
portfolio_returns = (
    gbm_final/S0 - 1
)

VaR_95 = np.percentile(
    portfolio_returns,
    5
)

print(
    "95% VaR:",
    VaR_95
)

There is a 5% chance that losses exceed VaR over the forecast horizon.

# **Section 14: Expected Shortfall**

In [ ]:
tail_returns = (
    portfolio_returns[
        portfolio_returns <= VaR_95
    ]
)

CVaR = tail_returns.mean()

print(
    "Expected Shortfall:",
    CVaR
)

# **Section 15: Compare Models**

In [ ]:
comparison = pd.DataFrame({
"Model":[
"GBM",
"OU",
"Jump-Diffusion"
],

"Mean Price":[
gbm_final.mean(),
ou_final.mean(),
jd_final.mean()
],

"Std Dev":[
gbm_final.std(),
ou_final.std(),
jd_final.std()
]
})

comparison

Section 16: Fan Chart

In [ ]:
percentiles = np.percentile(
    gbm_paths,
    [5,25,50,75,95],
    axis=1
)

plt.figure(figsize=(12,6))

plt.fill_between(
    range(DAYS),
    percentiles[0],
    percentiles[4],
    alpha=.2
)

plt.fill_between(
    range(DAYS),
    percentiles[1],
    percentiles[3],
    alpha=.4
)

plt.plot(
    percentiles[2],
    linewidth=2
)

plt.title("GBM Forecast Fan Chart")
plt.show()